# Reinforcement Learning aplicado a Hardware
## Taller GDG - Aprendiendo a entrenar agentes con Gymnasium

---

En este taller vamos a aprender los fundamentos de **Reinforcement Learning (RL)** de forma practica, entrenando agentes en entornos simulados que representan problemas reales de control fisico.

### Que vamos a ver?

| Entorno | Dificultad | Que aprenderemos |
|---------|-----------|-----------------|
| **CartPole** | Basica | Conceptos fundamentales de RL, Gymnasium API |
| **MountainCar** | Media | El problema de exploracion vs explotacion |
| **Acrobot** | Media-Alta | Sistemas articulados, recompensas sparse |
| **Pendulum** | Alta | Control continuo (acciones no son 0/1, son valores reales) |

### Por que esto importa para hardware?

Todos estos entornos simulan **problemas de control fisico**: equilibrar un palo, balancear articulaciones, controlar un pendulo... Son exactamente el tipo de problemas que aparecen en robotica, drones, y sistemas embebidos. La idea es: **entrenas en simulacion, despliegas en hardware real**.

## Parte 0: Que es Reinforcement Learning?

A diferencia de otros tipos de Machine Learning, en RL no tenemos un dataset con etiquetas. Tenemos un **agente** que interactua con un **entorno** y aprende por prueba y error.

```
    +------------------+
    |     ENTORNO      |
    |  (el mundo real  |
    |   o simulado)    |
    +--------+---------+
             |
             | Estado (s_t) + Recompensa (r_t)
             v
    +--------+---------+
    |      AGENTE      |
    |  (red neuronal   |
    |   o algoritmo)   |
    +--------+---------+
             |
             | Accion (a_t)
             v
    +------------------+
    |     ENTORNO      |
    |  ejecuta accion  |
    |  devuelve nuevo  |
    |  estado y reward  |
    +------------------+
```

### Conceptos clave

- **Estado (State)**: Lo que el agente observa del entorno (posicion, velocidad, angulos...)
- **Accion (Action)**: Lo que el agente puede hacer (moverse izquierda/derecha, aplicar fuerza...)
- **Recompensa (Reward)**: Senal numerica que indica si la accion fue buena o mala
- **Politica (Policy)**: La estrategia del agente - dado un estado, que accion tomar
- **Episodio**: Una "partida" completa desde el inicio hasta que el agente termina o falla

### El objetivo

Encontrar la **politica optima**: la estrategia que maximiza la **recompensa acumulada** a lo largo del tiempo.

## Parte 0.5: PPO - El algoritmo que usaremos

**Proximal Policy Optimization (PPO)** es uno de los algoritmos mas populares y robustos de RL. Es el algoritmo por defecto en muchas aplicaciones (OpenAI lo uso para entrenar modelos de chat, por ejemplo).

### Por que PPO?

- **Estable**: No colapsa facilmente durante el entrenamiento
- **Versatil**: Funciona con acciones discretas (izquierda/derecha) y continuas (aplicar X newtons de fuerza)
- **Eficiente**: Buen balance entre calidad de la politica y velocidad de entrenamiento

### Como funciona (version simplificada)

1. El agente juega varias partidas con su politica actual
2. Calcula que acciones fueron mejores de lo esperado (ventaja positiva) y cuales peores (ventaja negativa)
3. Actualiza la politica para hacer mas probables las buenas acciones y menos probables las malas
4. **Truco clave de PPO**: limita cuanto puede cambiar la politica en cada actualizacion (el "proximal" del nombre). Esto evita que el agente "desaprenda" de golpe lo que ya sabia

Usaremos la implementacion de **Stable Baselines 3**, que nos da PPO listo para usar.

In [ ]:
# Setup del entorno (ejecutar solo una vez)
# Necesitas tener uv instalado: pip install uv
# Luego desde la terminal, en la carpeta del proyecto:
#   uv sync
# Esto crea el .venv e instala todas las dependencias automaticamente

In [ ]:
import os
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import torch

from stable_baselines3 import PPO, A2C
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold
from stable_baselines3.common.monitor import Monitor

# Detectar dispositivo
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Dispositivo: {device}")

In [ ]:
def record_episode(env_name, model=None, max_steps=500):
    """Graba un episodio como secuencia de frames para visualizar en el notebook.
    Si model=None, usa acciones aleatorias."""
    env = gym.make(env_name, render_mode="rgb_array")
    frames = []
    obs, info = env.reset()
    
    for _ in range(max_steps):
        frames.append(env.render())
        if model is not None:
            action, _ = model.predict(obs, deterministic=True)
        else:
            action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        if terminated or truncated:
            break
    
    env.close()
    return frames


def show_animation(frames, interval=50):
    """Muestra una animacion inline en el notebook a partir de frames."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis("off")
    img = ax.imshow(frames[0])
    
    def update(frame):
        img.set_data(frame)
        return [img]
    
    anim = animation.FuncAnimation(fig, update, frames=frames, interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


def plot_training_curve(log_dir, title="Curva de entrenamiento"):
    """Grafica la recompensa por episodio desde los logs del Monitor."""
    from stable_baselines3.common.results_plotter import load_results, ts2xy
    x, y = ts2xy(load_results(log_dir), "timesteps")
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, y, alpha=0.3, color="blue", label="Por episodio")
    
    # Media movil
    window = max(1, len(y) // 20)
    if len(y) >= window:
        moving_avg = np.convolve(y, np.ones(window)/window, mode="valid")
        ax.plot(x[window-1:], moving_avg, color="red", linewidth=2, label=f"Media movil ({window} eps)")
    
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Recompensa")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()



---

# PARTE 1: CartPole - El "Hello World" de RL

<img src="https://gymnasium.farama.org/_images/cart_pole.gif" width="400">

### El problema
Un carro se mueve sobre un riel y tiene un palo encima. El objetivo es **mantener el palo en equilibrio** moviendo el carro a izquierda o derecha.

### Analogia hardware
Piensa en un **robot equilibrista** o un **drone estabilizandose**. El controlador (nuestro agente) tiene que hacer ajustes constantes y rapidos para mantener el equilibrio.

### Especificaciones del entorno

| Concepto | Detalle |
|----------|---------|
| **Observacion** | 4 valores: posicion del carro, velocidad del carro, angulo del palo, velocidad angular del palo |
| **Acciones** | 2 discretas: empujar izquierda (0) o derecha (1) |
| **Recompensa** | +1 por cada paso que el palo sigue en pie |
| **Termina cuando** | El palo cae mas de 12 grados, o el carro sale del riel, o se llega a 500 pasos |
| **Objetivo** | Conseguir recompensa de 500 (mantener el palo 500 pasos seguidos) |

### 1.1 Explorando el entorno - Acciones aleatorias

Antes de entrenar nada, veamos como se comporta el entorno con un agente que toma **acciones completamente aleatorias**. Esto nos da una linea base: el peor rendimiento posible.

In [ ]:
# Veamos la estructura del entorno
env = gym.make("CartPole-v1")
print(f"Espacio de observacion: {env.observation_space}")
print(f"  -> Forma: {env.observation_space.shape}")
print(f"  -> Limites: [{env.observation_space.low}] a [{env.observation_space.high}]")
print(f"\nEspacio de acciones: {env.action_space}")
print(f"  -> Numero de acciones: {env.action_space.n}")
print(f"\nEjemplo de observacion inicial:")
obs, info = env.reset()
print(f"  -> {obs}")
print(f"  -> [posicion_carro, velocidad_carro, angulo_palo, velocidad_angular_palo]")
env.close()

In [ ]:
# Agente aleatorio - nuestra linea base
frames_random_cartpole = record_episode("CartPole-v1", model=None)
print(f"El agente aleatorio sobrevivio {len(frames_random_cartpole)} pasos (de 500 posibles)")
show_animation(frames_random_cartpole)

### 1.2 Entrenando un agente con PPO

Ahora entrenemos un agente de verdad. Con PPO y solo 20,000 pasos de entrenamiento (menos de un minuto), el agente deberia aprender a mantener el palo en equilibrio.

In [ ]:
# Creamos el entorno con Monitor para registrar las recompensas
log_dir = "logs/cartpole"
os.makedirs(log_dir, exist_ok=True)

env = Monitor(gym.make("CartPole-v1"), log_dir)
env = DummyVecEnv([lambda: env])

# Creamos el modelo PPO
model_cartpole = PPO(
    "MlpPolicy",       # Red neuronal fully-connected (Multi-Layer Perceptron)
    env,
    device=device,
    verbose=1,
    learning_rate=3e-4, # Tasa de aprendizaje (default de PPO)
    n_steps=2048,       # Pasos antes de cada actualizacion
    batch_size=64,      # Tamano del mini-batch
)

print("Arquitectura de la red neuronal del agente:")
print(model_cartpole.policy)

In [ ]:
# Entrenamos! 20k timesteps son suficientes para CartPole
model_cartpole.learn(total_timesteps=20_000)
print("Entrenamiento completado!")

In [ ]:
# Visualizamos la curva de entrenamiento
plot_training_curve(log_dir, "CartPole - Curva de aprendizaje")

In [ ]:
# Evaluacion cuantitativa
env_eval = gym.make("CartPole-v1")
mean_reward, std_reward = evaluate_policy(model_cartpole, env_eval, n_eval_episodes=10)
print(f"Recompensa media: {mean_reward:.1f} +/- {std_reward:.1f}  (objetivo: 500)")
env_eval.close()

In [ ]:
# Veamos al agente entrenado en accion!
frames_trained_cartpole = record_episode("CartPole-v1", model=model_cartpole)
print(f"El agente entrenado sobrevivio {len(frames_trained_cartpole)} pasos (de 500 posibles)")
show_animation(frames_trained_cartpole)

---

# PARTE 2: MountainCar - El problema de la exploracion

<img src="https://gymnasium.farama.org/_images/mountain_car.gif" width="400">

### El problema
Un coche esta atrapado en un valle entre dos montanas. El motor no es lo suficientemente potente para subir directamente. Tiene que **coger impulso** balanceandose de un lado a otro.

### Por que es interesante?
Este entorno ilustra uno de los problemas fundamentales de RL: la **exploracion**. El agente recibe -1 de recompensa en cada paso. No hay recompensa positiva hasta que llega a la cima. Si el agente solo hace cosas que dan "buena" recompensa inmediata... nunca descubrira que balancearse es la solucion.

Es como si le dijeras a alguien "cada segundo que pases aqui te cuesta 1 euro" pero no le dices que hay una salida. Tiene que **explorar** para encontrarla.

### Analogia hardware
Imagina un robot que tiene que superar un obstaculo mas alto que su capacidad de salto directo. Necesita encontrar una estrategia no obvia (coger carrerilla, usar inercia...).

### Especificaciones

| Concepto | Detalle |
|----------|---------|
| **Observacion** | 2 valores: posicion (-1.2 a 0.6) y velocidad (-0.07 a 0.07) |
| **Acciones** | 3 discretas: empujar izquierda (0), no hacer nada (1), empujar derecha (2) |
| **Recompensa** | -1 por cada paso (penalizacion por tardar) |
| **Termina cuando** | Llega a la bandera (posicion >= 0.5) o pasan 200 pasos |
| **Objetivo** | Llegar a la bandera en el menor numero de pasos posible |

In [ ]:
# Agente aleatorio en MountainCar - probablemente no llegue nunca
frames_random_mc = record_episode("MountainCar-v0", model=None, max_steps=200)
print(f"Pasos: {len(frames_random_mc)} (si llego a 200, no consiguio subir)")
show_animation(frames_random_mc)

### 2.1 El truco: Reward Shaping

MountainCar es famosamente dificil para PPO con la recompensa por defecto (-1 por paso). El agente nunca recibe una senal positiva porque nunca llega arriba por casualidad.

**Solucion**: Modificamos la recompensa para guiar al agente. Le damos recompensa extra por **ganar altura y velocidad**. Esto se llama **reward shaping** y es una tecnica fundamental en RL aplicado.

In [ ]:
class MountainCarShaped(gym.Wrapper):
    """Wrapper que modifica la recompensa de MountainCar para facilitar el aprendizaje."""
    
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        position, velocity = obs
        
        # Recompensa extra por estar alto (la meta esta en posicion 0.5)
        reward += position * 10
        # Recompensa extra por tener velocidad (necesita impulso)
        reward += abs(velocity) * 100
        # Gran bonus por llegar a la meta
        if terminated and position >= 0.5:
            reward += 1000
        
        return obs, reward, terminated, truncated, info


# Entrenamos con reward shaping
log_dir_mc = "logs/mountaincar"
os.makedirs(log_dir_mc, exist_ok=True)

env_mc = Monitor(MountainCarShaped(gym.make("MountainCar-v0")), log_dir_mc)
env_mc = DummyVecEnv([lambda: env_mc])

model_mc = PPO("MlpPolicy", env_mc, device=device, verbose=1, learning_rate=3e-4)
model_mc.learn(total_timesteps=100_000)
print("Entrenamiento completado!")

In [ ]:
# Veamos si aprendio a subir la montana
frames_trained_mc = record_episode("MountainCar-v0", model=model_mc, max_steps=200)
print(f"Pasos para llegar: {len(frames_trained_mc)} (menos es mejor, max 200)")
show_animation(frames_trained_mc)

---

# PARTE 3: Acrobot - Un robot articulado

<img src="https://gymnasium.farama.org/_images/acrobot.gif" width="300">

### El problema
El Acrobot es un robot de **dos articulaciones** (como un brazo con codo). La primera articulacion esta fija al techo. El objetivo es balancear el extremo del segundo segmento **por encima de una linea horizontal**.

El truco: solo puedes aplicar torque en la articulacion del medio (el "codo"), no en la base. Es como intentar hacer un salto mortal en una barra de gimnasia: tienes que usar el impulso de tu cuerpo.

### Analogia hardware
- **Brazos roboticos**: muchos robots industriales tienen articulaciones donde no todas son actuadas directamente
- **Exoesqueletos**: sistemas donde hay que coordinar multiples articulaciones con actuadores limitados
- **Robots blandos**: estructuras flexibles que se mueven de formas no intuitivas

### Especificaciones

| Concepto | Detalle |
|----------|---------|
| **Observacion** | 6 valores: cos(theta1), sin(theta1), cos(theta2), sin(theta2), velocidad angular 1, velocidad angular 2 |
| **Acciones** | 3 discretas: torque -1 (0), torque 0 (1), torque +1 (2) |
| **Recompensa** | -1 por cada paso hasta que el extremo supera la linea |
| **Termina cuando** | El extremo del segundo segmento supera la linea, o pasan 500 pasos |
| **Objetivo** | Llegar arriba en el menor numero de pasos posible |

### Por que es interesante pedagogicamente?

Al igual que MountainCar, la recompensa es **sparse** (solo -1 por paso). Pero aqui el espacio de observacion es mas rico (6D vs 2D) y el movimiento es mas complejo porque involucra **dos cuerpos rigidos acoplados**. El agente tiene que descubrir que necesita balancearse para acumular energia, algo nada obvio.

In [ ]:
# Agente aleatorio en Acrobot - se balancea sin sentido
frames_random_acro = record_episode("Acrobot-v1", model=None, max_steps=500)
print(f"Pasos: {len(frames_random_acro)} (si llego a 500, no consiguio subir)")
show_animation(frames_random_acro)

### 3.1 Entrenando al Acrobot

A pesar de tener recompensa sparse, Acrobot es mas facil que MountainCar porque el espacio de observacion es mas informativo (6 valores vs 2). PPO puede resolverlo razonablemente con **100,000 timesteps**.

In [ ]:
log_dir_acro = "logs/acrobot"
os.makedirs(log_dir_acro, exist_ok=True)

env_acro = Monitor(gym.make("Acrobot-v1"), log_dir_acro)
env_acro = DummyVecEnv([lambda: env_acro])

model_acro = PPO(
    "MlpPolicy",
    env_acro,
    device=device,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
)

model_acro.learn(total_timesteps=100_000)
print("Entrenamiento completado!")

In [ ]:
# Curva de aprendizaje
plot_training_curve(log_dir_acro, "Acrobot - Curva de aprendizaje")

In [ ]:
# Evaluacion
env_eval_acro = gym.make("Acrobot-v1")
mean_reward, std_reward = evaluate_policy(model_acro, env_eval_acro, n_eval_episodes=10)
print(f"Recompensa media: {mean_reward:.1f} +/- {std_reward:.1f}  (mas cerca de 0 = mejor, -500 = nunca llego)")
env_eval_acro.close()

In [ ]:
# Veamos al Acrobot entrenado balancearse hasta arriba
frames_trained_acro = record_episode("Acrobot-v1", model=model_acro, max_steps=500)
print(f"Llego en {len(frames_trained_acro)} pasos (menos es mejor, max 500)")
show_animation(frames_trained_acro)

### 3.2 Comparando antes y despues

Veamos las estadisticas del agente aleatorio vs el entrenado.

In [ ]:
# Comparacion cuantitativa: aleatorio vs entrenado
def evaluate_random(env_name, n_episodes=20):
    env = gym.make(env_name)
    rewards = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        total_reward = 0
        done = False
        while not done:
            obs, reward, terminated, truncated, _ = env.step(env.action_space.sample())
            total_reward += reward
            done = terminated or truncated
        rewards.append(total_reward)
    env.close()
    return np.mean(rewards), np.std(rewards)

random_mean, random_std = evaluate_random("Acrobot-v1")
trained_mean, trained_std = evaluate_policy(model_acro, gym.make("Acrobot-v1"), n_eval_episodes=20)

fig, ax = plt.subplots(figsize=(8, 4))
agents = ["Aleatorio", "PPO Entrenado"]
means = [random_mean, trained_mean]
stds = [random_std, trained_std]
colors = ["#e74c3c", "#2ecc71"]

bars = ax.bar(agents, means, yerr=stds, color=colors, capsize=10, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Recompensa media (mas cerca de 0 = mejor)")
ax.set_title("Acrobot: Aleatorio vs Entrenado")

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f"{mean:.0f}", 
            ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

---

# PARTE 4: Pendulum - Control continuo

<img src="https://gymnasium.farama.org/_images/pendulum.gif" width="300">

### El salto a acciones continuas

Hasta ahora, todas nuestras acciones eran **discretas**: izquierda o derecha, motor encendido o apagado. Pero en el mundo real del hardware, las acciones suelen ser **continuas**: no dices "motor ON" o "motor OFF", dices "aplica 2.7 voltios" o "gira 37.5 grados".

### El problema
Un pendulo cuelga de un punto fijo. El objetivo es **balancearlo hasta que quede vertical hacia arriba** y mantenerlo ahi. La unica accion es aplicar un **torque** (fuerza de rotacion) que puede ir de -2.0 a +2.0.

### Por que es relevante para hardware?
Esto es **exactamente** lo que hace un controlador de motor. Piensa en:
- Un brazo robotico que tiene que mantener una posicion
- Un gimbal de camara estabilizando la imagen
- Un servo que tiene que moverse a una posicion exacta con suavidad

### Especificaciones

| Concepto | Detalle |
|----------|---------|
| **Observacion** | 3 valores: cos(angulo), sin(angulo), velocidad angular |
| **Acciones** | 1 valor **continuo**: torque entre -2.0 y +2.0 |
| **Recompensa** | -(angulo^2 + 0.1*velocidad^2 + 0.001*torque^2). Maximo 0 cuando esta vertical y quieto |
| **Termina cuando** | Nunca termina por si solo (se trunca a 200 pasos) |
| **Objetivo** | Maximizar recompensa (acercarse a 0, ya que siempre es negativa) |

**Nota clave**: La recompensa siempre es negativa. El "mejor" resultado posible es 0 (pendulo perfectamente vertical, sin moverse, sin aplicar fuerza). Esto es tipico en problemas de control: minimizar el error.

In [ ]:
# Veamos la diferencia en el espacio de acciones
env_discrete = gym.make("CartPole-v1")
env_continuous = gym.make("Pendulum-v1")

print("=== CartPole (DISCRETO) ===")
print(f"  Tipo: {type(env_discrete.action_space).__name__}")
print(f"  Acciones posibles: {env_discrete.action_space.n}")
print(f"  Ejemplo: {env_discrete.action_space.sample()}")

print(f"\n=== Pendulum (CONTINUO) ===")
print(f"  Tipo: {type(env_continuous.action_space).__name__}")
print(f"  Rango: [{env_continuous.action_space.low}] a [{env_continuous.action_space.high}]")
print(f"  Ejemplo: {env_continuous.action_space.sample()}")
print(f"  Ejemplo: {env_continuous.action_space.sample()}")
print(f"  Ejemplo: {env_continuous.action_space.sample()}")

env_discrete.close()
env_continuous.close()

In [ ]:
# Pendulo con acciones aleatorias - gira sin control
frames_random_pend = record_episode("Pendulum-v1", model=None, max_steps=200)
show_animation(frames_random_pend, interval=50)

### 4.1 Entrenando el controlador

Para acciones continuas, PPO funciona pero hay que darle mas tiempo. La red neuronal ahora no elige entre opciones discretas sino que produce un valor continuo (usando una distribucion gaussiana).

In [ ]:
log_dir_pend = "logs/pendulum"
os.makedirs(log_dir_pend, exist_ok=True)

env_pend = Monitor(gym.make("Pendulum-v1"), log_dir_pend)
env_pend = DummyVecEnv([lambda: env_pend])

model_pend = PPO(
    "MlpPolicy",
    env_pend,
    device=device,
    verbose=1,
    learning_rate=1e-3,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    gamma=0.95,         # Descuento mas bajo: el futuro cercano importa mas
    gae_lambda=0.9,
)

model_pend.learn(total_timesteps=200_000)
print("Entrenamiento completado!")

In [ ]:
# Curva de aprendizaje del Pendulum
plot_training_curve(log_dir_pend, "Pendulum - Curva de aprendizaje")

In [ ]:
# Veamos al pendulo entrenado
frames_trained_pend = record_episode("Pendulum-v1", model=model_pend, max_steps=200)
show_animation(frames_trained_pend, interval=50)

---

# PARTE 5: Resumen y vision general

### Lo que hemos aprendido

| Entorno | Concepto clave | Tipo de control |
|---------|---------------|-----------------|
| **CartPole** | Loop basico de RL, API de Gymnasium | Discreto (2 acciones) |
| **MountainCar** | Exploracion, reward shaping | Discreto (3 acciones) |
| **Acrobot** | Sistemas articulados, recompensas sparse | Discreto (3 acciones) |
| **Pendulum** | Acciones continuas, control de torque | Continuo (valor real) |

### De la simulacion al hardware real

El flujo de trabajo en RL aplicado a hardware sigue estos pasos:

```
1. SIMULAR    ->  2. ENTRENAR     ->  3. TRANSFERIR   ->  4. AJUSTAR
Crear entorno     Entrenar agente     Desplegar en        Fine-tuning
en Gymnasium      con PPO/SAC/etc     hardware real       en el dispositivo
```

**Sim-to-Real Transfer** es el campo que estudia como pasar modelos entrenados en simulacion a dispositivos reales. Los principales retos son:
- La simulacion nunca es perfecta (friccion, ruido, delays)
- Se usan tecnicas como **Domain Randomization** (variar parametros de la simulacion durante el entrenamiento)
- Frameworks como **NVIDIA Isaac Sim** o **MuJoCo** ofrecen simulaciones fisicas muy realistas

### Retos para seguir practicando

1. **Facil**: Cambia los hiperparametros de CartPole (learning_rate, n_steps) y observa como afecta la curva de entrenamiento
2. **Medio**: Entrena Acrobot con A2C en vez de PPO (`from stable_baselines3 import A2C`) y compara resultados
3. **Dificil**: Intenta resolver MountainCar **sin** reward shaping, usando un algoritmo diferente o mas timesteps
4. **Avanzado**: Crea tu propio entorno de Gymnasium (un brazo robotico simple, un termostato...) siguiendo la [documentacion oficial](https://gymnasium.farama.org/tutorials/gymnasium_basics/environment_creation/)

### Recursos

- [Gymnasium Docs](https://gymnasium.farama.org/) - Documentacion oficial de los entornos
- [Stable Baselines 3 Docs](https://stable-baselines3.readthedocs.io/) - Documentacion de los algoritmos
- [Spinning Up in Deep RL (OpenAI)](https://spinningup.openai.com/) - Curso gratuito de RL, excelente para profundizar
- [CleanRL](https://github.com/vwxyzjn/cleanrl) - Implementaciones limpias de algoritmos RL en un solo archivo

In [ ]:
# Guardamos todos los modelos entrenados
os.makedirs("models", exist_ok=True)
model_cartpole.save("models/ppo_cartpole")
model_mc.save("models/ppo_mountaincar")
model_acro.save("models/ppo_acrobot")
model_pend.save("models/ppo_pendulum")
print("Modelos guardados en ./models/")